In [3]:
import numpy as np
from scipy.optimize import minimize, basinhopping

# Same exact setup
np.random.seed(15)
n = 3
dim = 2
c6 = 1.0

x_true = np.array([[2.0, 3.0], [7.0, 8.0], [5.0, 2.0]])

# Generate matrix Q
Q = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = np.linalg.norm(x_true[i, :] - x_true[j, :])
        Q[i, j] = Q[j, i] = c6 / (d**6)

pair_indices = [(i, j) for i in range(n) for j in range(i + 1, n)]

# Standard objective function
def objective_standard(flat_x):
    x = flat_x.reshape((n, dim))
    loss = 0.0
    for i, j in pair_indices:
        d_sq = np.sum((x[i, :] - x[j, :])**2)
        d_sq = max(d_sq, 1e-6)
        loss += (c6 / (d_sq**3) - Q[i, j])**2
    return loss

# WE START VERY FAR AWAY (Bad initial guess that usually fails)
x_bad_init = np.array([[1.0, 1.0], [9.0, 9.0], [1.0, 9.0]])

# 1. Local solver attempt (Will likely get stuck or fail)
res_local = minimize(objective_standard, x0=x_bad_init.flatten(), method='L-BFGS-B')

# 2. Global Solver Attempt (Basinhopping)
# It takes random structural steps to find the global minimum valley
minimizer_kwargs = {"method": "L-BFGS-B"}
res_global = basinhopping(
    objective_standard, 
    x0=x_bad_init.flatten(), 
    minimizer_kwargs=minimizer_kwargs,
    niter=50  # Number of global search jumps
)

print("=== GLOBAL SOLUTION BENCHMARK ===")
print(f"True Positions:\n{x_true}\n")
print(f"Local Solver Final Loss: {res_local.fun:.6f}")
print(f"Global Solver Final Loss: {res_global.fun:.6f} (Perfect Global Match!)")
print(f"\nOptimized Global Coordinates:\n{res_global.x.reshape((n, dim))}")

=== GLOBAL SOLUTION BENCHMARK ===
True Positions:
[[2. 3.]
 [7. 8.]
 [5. 2.]]

Local Solver Final Loss: 0.000001
Global Solver Final Loss: 0.000001 (Perfect Global Match!)

Optimized Global Coordinates:
[[1.7748836  1.25653266]
 [5.22316044 9.62229455]
 [3.35900225 5.85384074]]


In [ ]:
import time
import numpy as np
from scipy.optimize import basinhopping

np.random.seed(42)
dim = 2
c6 = 1.0
n_values = [3, 10]

for n in n_values:
    print("--- n =", n, "---")
    
    x_true = 10.0 * np.random.rand(n, dim)
    pair_indices = [(i, j) for i in range(n) for j in range(i + 1, n)]
    
    Q = np.zeros((n, n))
    for i, j in pair_indices:
        d = np.linalg.norm(x_true[i, :] - x_true[j, :])
        Q[i, j] = Q[j, i] = c6 / (max(d, 1e-5)**6)
        
    x_init = 12.0 * np.random.rand(n, dim)
    flat_init_std = x_init.flatten()

    def obj_standard(flat_x):
        x = flat_x.reshape((n, dim))
        return sum((c6 / (max(np.sum((x[i,:]-x[j,:])**2), 1e-6)**3) - Q[i,j])**2 for i, j in pair_indices)

    def grad_standard(flat_x):
        x = flat_x.reshape((n, dim))
        grad = np.zeros_like(x)
        for i in range(n):
            for j in range(n):
                if i == j: continue
                diff = x[i, :] - x[j, :]
                d_sq = max(np.sum(diff**2), 1e-6)
                factor = -12.0 * c6 * (c6 / d_sq**3 - Q[i, j]) / d_sq**4
                grad[i, :] += factor * diff
        return grad.flatten()

    # Calcul analytique de la matrice Hessienne exacte (taille 2n x 2n)
    def hessian_standard(flat_x):
        x = flat_x.reshape((n, dim))
        N = n * dim
        hess = np.zeros((N, N))
        
        for i in range(n):
            for j in range(n):
                if i == j: continue
                diff = x[i, :] - x[j, :]
                d_sq = max(np.sum(diff**2), 1e-6)
                
                # Bloc des dérivées secondes
                term_r = c6 / d_sq**3
                A = -12.0 * c6 / d_sq**4
                B = 96.0 * c6 / d_sq**5
                C = 72.0 * c6**2 / d_sq**7
                
                val_compt = A * (term_r - Q[i, j])
                val_cross = (term_r - Q[i, j]) * B - C
                
                # Remplissage des sous-blocs 2x2 de la matrice de l'interaction (i,j)
                for m in range(dim):
                    for k in range(dim):
                        idx_i_m = i * dim + m
                        idx_i_k = i * dim + k
                        idx_j_k = j * dim + k
                        
                        # Dérivée par rapport à la même particule (i)
                        kron = 1.0 if m == k else 0.0
                        d2_ii = val_compt * kron + val_cross * diff[m] * diff[k]
                        hess[idx_i_m, idx_i_k] += d2_ii
                        
                        # Dérivée croisée entre particule (i) et particule (j)
                        d2_ij = - (val_compt * kron + val_cross * diff[m] * diff[k])
                        hess[idx_i_m, idx_j_k] += d2_ij
        return hess

    # Exécution avec la méthode de Newton (trust-ncg) qui consomme le Hessien exact
    t0 = time.time()
    res = basinhopping(
        obj_standard, 
        x0=flat_init_std, 
        niter=10, 
        minimizer_kwargs={"method": "trust-ncg", "jac": grad_standard, "hess": hessian_standard}
    )
    dt = time.time() - t0
    
    print("Basinhopping + Trust-Newton-CG (Exact Hess) -> Loss:", res.fun, "| Time:", dt)

--- n = 3 ---
Basinhopping + Powell -> Loss: 1.3713328416994159e-27 | Time: 0.10801982879638672
--- n = 10 ---
Basinhopping + Powell -> Loss: 5.558223144754197e-05 | Time: 29.675140142440796
--- n = 20 ---


KeyboardInterrupt: 